# 04 · International tournaments (World Cup)

**Goal:** for each season and nation, how far it got in the World Cup (`wc_stage`, 0–6), to fill the blank WC rows in `data/manual/intl_tournaments.csv`.

**Stage scale:** 6 winner · 5 runner-up · 4 semis · 3 quarters · 2 round of 16 · 1 group stage · 0 didn't play

**Season rule:** World Cups 2010, 2014, 2018 map to the same season. The 2022 World Cup (Nov–Dec) maps to **season 2023**, because the 2022 award was given before it. The 2026 World Cup is set aside for the prediction set.

**Source:** Wikipedia (`List_of_FIFA_World_Cup_finals` + each `{year}_FIFA_World_Cup` page).

In [1]:
import pandas as pd
import requests
from io import StringIO

headers = {"User-Agent": "Mozilla/5.0"}

## 1. Finals: winners (6) and runners-up (5)

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals"
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text), match="Winners")

len(tables)

3

In [3]:
wc_champions = tables[0]
wc_champions = wc_champions[["Year", "Winners", "Runners-up"]]
wc_champions.columns = ["year", "winner", "runner_up"]
wc_champions

,year,winner,runner_up
0,1930,Uruguay,Argentina
1,1934,Italy,Czechoslovakia
2,1938,Italy,Hungary
3,1950,Uruguay,Brazil
4,1954,West Germany,Hungary
5,1958,Brazil,Sweden
6,1962,Brazil,Czechoslovakia
7,1966,England,West Germany
8,1970,Brazil,Italy
9,1974,West Germany,Netherlands


In [4]:
wc_champions = wc_champions[wc_champions["year"].between(2010, 2025)]
wc_champions

,year,winner,runner_up
18,2010,Spain,Netherlands
19,2014,Germany,Argentina
20,2018,France,Croatia
21,2022,Argentina,France


In [5]:
winners = wc_champions[["year", "winner"]].copy()

w = winners[["year","winner"]].rename(columns={"winner":"nation"})
w['wc_stage'] = 6
w

,year,nation,wc_stage
18,2010,Spain,6
19,2014,Germany,6
20,2018,France,6
21,2022,Argentina,6


In [6]:
runners_up = wc_champions[["year", "runner_up"]].copy()

ru = runners_up[["year", "runner_up"]].rename(columns={"runner_up":"nation"})
ru['wc_stage'] = 5
ru

,year,nation,wc_stage
18,2010,Netherlands,5
19,2014,Argentina,5
20,2018,Croatia,5
21,2022,France,5


In [7]:
wc = pd.concat([w, ru], ignore_index=True)

wc

,year,nation,wc_stage
0,2010,Spain,6
1,2014,Germany,6
2,2018,France,6
3,2022,Argentina,6
4,2010,Netherlands,5
5,2014,Argentina,5
6,2018,Croatia,5
7,2022,France,5


## 2. Knockout bracket: 2018 (prototype)

The bracket comes as a wide table: each round is a column pair. Columns **1 = round of 16, 5 = quarters, 9 = semis, 13 = final**. Noise to remove: dates (contain `–`), the round header, repeated names, and tags like `(p)` / `(a.e.t.)`.

In [8]:
url_semi = "https://en.wikipedia.org/wiki/2018_FIFA_World_Cup#Knockout_stage"
response_semi = requests.get(url_semi, headers=headers)
tables_semi = pd.read_html(StringIO(response_semi.text), match="Semi-finals")

len(tables_semi)

2

In [9]:
bracket = tables_semi[0]

In [10]:
r16 = bracket[1].dropna()
r16 = r16[~r16.str.contains("–")]
r16 = r16[r16 != "Round of 16"]
r16 = r16.str.replace(r"\s*\(.*\)", "", regex=True).drop_duplicates()
r16

4         Uruguay
6        Portugal
10         France
12      Argentina
16         Brazil
18         Mexico
22        Belgium
24          Japan
28          Spain
30         Russia
34        Croatia
36        Denmark
40         Sweden
42    Switzerland
46       Colombia
48        England
Name: 1, dtype: object

In [11]:
qf = bracket[5].dropna()
qf = qf[~qf.str.contains("–")]
qf = qf[qf != "Quarter-finals"]
qf = qf.str.replace(r"\s*\(.*\)", "", regex=True).drop_duplicates()
qf

7     Uruguay
9      France
19     Brazil
21    Belgium
31     Russia
33    Croatia
43     Sweden
45    England
Name: 5, dtype: object

In [12]:
sf = bracket[9].dropna()
sf = sf[~sf.str.contains("–")]
sf = sf[sf != "Semi-finals"]
sf = sf.str.replace(r"\s*\(.*\)", "", regex=True).drop_duplicates()
sf

13     France
15    Belgium
37    Croatia
39    England
Name: 9, dtype: object

### Long format + keep the highest stage

A nation shows up once per round it reached, so `groupby(...).max()` keeps how far it got.

**Check:** 2018 has 16 nations: one 6, one 5, two 4s, four 3s, eight 2s.

In [13]:
r16_df = pd.DataFrame({"year": 2018, "nation": r16.values, "wc_stage": 2})
qf_df = pd.DataFrame({"year": 2018, "nation": qf.values, "wc_stage": 3})
sf_df = pd.DataFrame({"year": 2018, "nation": sf.values, "wc_stage": 4})

In [14]:
wc_standing = pd.concat([wc, sf_df, qf_df, r16_df], ignore_index=True)
wc_standing


,year,nation,wc_stage
0,2010,Spain,6
1,2014,Germany,6
2,2018,France,6
3,2022,Argentina,6
4,2010,Netherlands,5
5,2014,Argentina,5
6,2018,Croatia,5
7,2022,France,5
8,2018,France,4
9,2018,Belgium,4


In [15]:
wc_standing = wc_standing.groupby(["year", "nation"], as_index=False)["wc_stage"].max()
wc_standing

,year,nation,wc_stage
0,2010,Netherlands,5
1,2010,Spain,6
2,2014,Argentina,5
3,2014,Germany,6
4,2018,Argentina,2
5,2018,Belgium,4
6,2018,Brazil,3
7,2018,Colombia,2
8,2018,Croatia,5
9,2018,Denmark,2


## 3. Other years (2010, 2014, 2022)

Checked: the bracket uses the same column layout (1/5/9) in every year.

### Cleaning function

Same 3 filters as section 2, but always starting from `bracket`, so rerunning it doesn't filter already-filtered data.

**Check:** `clean_round(bracket_2018, 5, "Quarter-finals")` → 8.

In [16]:
def clean_round(bracket, col, header):
    s = bracket[col].dropna()
    s = s[~s.str.contains("–")]
    s = s[s != header]
    s = s.str.replace(r"\s*\(.*\)", "", regex=True).drop_duplicates()
    return s

In [17]:
frames = []

for year in [2010, 2014, 2022]:
    # url with the year, read_html, clean columns 1/5/9, build the dfs
    url = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup#Knockout_stage"
    response = requests.get(url, headers=headers)
    tables = pd.read_html(StringIO(response.text), match="Semi-finals")
    bracket = tables[0]
    r16 = clean_round(bracket, 1, "Round of 16")
    qf = clean_round(bracket, 5, "Quarter-finals")
    sf = clean_round(bracket, 9, "Semi-finals")
    frames.append(pd.DataFrame({"year": year, "nation": r16.values, "wc_stage": 2}))
    frames.append(pd.DataFrame({"year": year, "nation": qf.values, "wc_stage": 3}))
    frames.append(pd.DataFrame({"year": year, "nation": sf.values, "wc_stage": 4}))


In [18]:
wc_standing = pd.concat([wc_standing] + frames, ignore_index=True)
wc_standing = wc_standing.groupby(["year", "nation"], as_index=False)["wc_stage"].max()
wc_standing["year"].value_counts()

year
2010    16
2014    16
2018    16
2022    16
Name: count, dtype: int64

In [19]:
wc_standing["year"] = wc_standing["year"].replace({2022: 2023})
wc_standing

,year,nation,wc_stage
0,2010,Argentina,3
1,2010,Brazil,3
2,2010,Chile,2
3,2010,England,2
4,2010,Germany,4
...,...,...,...
59,2023,Senegal,2
60,2023,South Korea,2
61,2023,Spain,2
62,2023,Switzerland,2


In [20]:
wc_standing['year'].value_counts()

year
2010    16
2014    16
2018    16
2023    16
Name: count, dtype: int64

In [21]:
nation_map = {
    "Argentina":"ARG","Brazil":"BRA","Chile":"CHI","England":"ENG","Germany":"GER",
    "Ghana":"GHA","Japan":"JPN","Mexico":"MEX","Netherlands":"NED","Paraguay":"PAR",
    "Portugal":"POR","Slovakia":"SVK","South Korea":"KOR","Spain":"ESP",
    "United States":"USA","Uruguay":"URU","Algeria":"ALG","Belgium":"BEL",
    "Colombia":"COL","Costa Rica":"CRC","France":"FRA","Greece":"GRE",
    "Nigeria":"NGA","Switzerland":"SUI","Croatia":"CRO","Denmark":"DEN",
    "Russia":"RUS","Sweden":"SWE","Australia":"AUS","Morocco":"MAR",
    "Poland":"POL","Senegal":"SEN"}
wc_standing["nation"] = wc_standing["nation"].map(nation_map)

In [22]:
wc_standing

,year,nation,wc_stage
0,2010,ARG,3
1,2010,BRA,3
2,2010,CHI,2
3,2010,ENG,2
4,2010,GER,4
...,...,...,...
59,2023,SEN,2
60,2023,KOR,2
61,2023,ESP,2
62,2023,SUI,2


In [23]:
wc_standing = wc_standing.rename(columns={"year": "season"})

In [24]:
intl = pd.read_csv("../data/manual/intl_tournaments.csv",
                   usecols=["season", "nation", "tournament", "intl_stage"])
intl = intl.merge(wc_standing, on=["season", "nation"], how="left")

In [25]:
intl

,season,nation,tournament,intl_stage,wc_stage
0,2010,ARG,WC,3,3.0
1,2010,BRA,WC,3,3.0
2,2010,CIV,WC,1,NaN
3,2010,CMR,WC,1,NaN
4,2010,ESP,WC,6,6.0
...,...,...,...,...,...
207,2025,NED,NaN,0,NaN
208,2025,NOR,NaN,0,NaN
209,2025,POL,NaN,0,NaN
210,2025,POR,NaN,0,NaN


## 4. Fill WC rows into intl_tournaments

Nations with no knockout result in a WC season: **1** = out in the group stage, **0** = didn't qualify (filled by hand, 19 rows, checked on Wikipedia).

This cell starts from the csv every time, so it's safe to rerun.

In [26]:
wc_seasons = [2010, 2014, 2018, 2023]

intl = pd.read_csv("../data/manual/intl_tournaments.csv",
                   usecols=["season", "nation", "tournament", "intl_stage"])
intl = intl.merge(wc_standing, on=["season", "nation"], how="left")

group_out = {(2010, "CIV"), (2010, "CMR"),
             (2014, "CIV"), (2014, "ESP"), (2014, "POR"),
             (2018, "EGY"), (2018, "SEN"),
             (2023, "BEL"), (2023, "CMR"), (2023, "GER")}

is_wc = intl["season"].isin(wc_seasons)
in_group = intl.apply(lambda r: (r["season"], r["nation"]) in group_out, axis=1)

intl.loc[is_wc & intl["wc_stage"].isna(), "wc_stage"] = 0
intl.loc[is_wc & in_group, "wc_stage"] = 1

intl.loc[is_wc, "intl_stage"] = intl.loc[is_wc, "wc_stage"]
intl.loc[is_wc & (intl["wc_stage"] > 0), "tournament"] = "WC"
intl = intl.drop(columns="wc_stage")
intl["intl_stage"] = intl["intl_stage"].astype(int)

print(intl.shape, intl["intl_stage"].isna().sum())
intl[is_wc]["intl_stage"].value_counts().sort_index()

(212, 4) 0


intl_stage
0     9
1    10
2     7
3    10
4     8
5     4
6     4
Name: count, dtype: int64

In [27]:
intl

,season,nation,tournament,intl_stage
0,2010,ARG,WC,3
1,2010,BRA,WC,3
2,2010,CIV,WC,1
3,2010,CMR,WC,1
4,2010,ESP,WC,6
...,...,...,...,...
207,2025,NED,NaN,0
208,2025,NOR,NaN,0
209,2025,POL,NaN,0
210,2025,POR,NaN,0


In [28]:
intl.to_csv("../data/manual/intl_tournaments.csv", index=False)

In [29]:
intl.shape

(212, 4)